# Preliminary Modeling

## Objectives
- Train preliminary models on both full and PCA-reduced features
    - Decision Tree Classifier
    - Random Forest
    - Gradient Boosting Classifier
    - Extra Trees Classifier
    - Logistic Regression
    - MLP Classifier
- Evaluate models using Precision, Recall, and F1 Score

In [ ]:
# Mount Google Drive (only needed in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set the workking directory (only needed in Google Colab)
import os
os.chdir('/content/drive/MyDrive/bitcoin-ransomware-classifier/bitcoin-ransomware-classifier/notebooks')
print('Working directory:', os.getcwd())

In [1]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")
X_train_pca = pd.read_csv("../data/X_train_pca.csv")
X_val = pd.read_csv("../data/X_val.csv")
X_val_pca = pd.read_csv("../data/X_val_pca.csv")
X_test = pd.read_csv("../data/X_test.csv")
X_test_pca = pd.read_csv("../data/X_test_pca.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_val = pd.read_csv("../data/y_val.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_train_pca shape:", X_train_pca.shape)
print("X_val shape:", X_val.shape)
print("X_val_pca shape:", X_val_pca.shape)
print("X_test shape:", X_test.shape)
print("X_test_pca shape:", X_test_pca.shape)

print("\ny_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

X_train shape: (4025396, 8)
X_train_pca shape: (4025396, 7)
X_val shape: (437505, 8)
X_val_pca shape: (437505, 7)
X_test shape: (437505, 8)
X_test_pca shape: (437505, 7)

y_train shape: (4025396,)
y_val shape: (437505,)
y_test shape: (437505,)


In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier as SkRandomForestClassifier

GPU_ENABLED = False

try:
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRandomForestClassifier
    GPU_ENABLED = True
    print("Using cuML RandomForestClassifier on GPU where available.")
except ImportError:
    cudf = None
    cuRandomForestClassifier = None
    print("cuML not available; using sklearn estimators on CPU.")

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as exc:
    XGBClassifier = None
    XGB_AVAILABLE = False
    print(f"XGBoost unavailable in this environment: {exc}")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/Ife/Documents/bitcoin-ransomware-classifier/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Users/Ife/Documents/bitcoin-ransomware-classifier/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot

## Modeling

In [ ]:
# cuML is taking over scaling here so this preprocessing step runs on the GPU in Colab; on local device this will fall back to CPU instead.
RandomForestClassifier = cuRandomForestClassifier if GPU_ENABLED else SkRandomForestClassifier

def prepare_features(model, X):
    if GPU_ENABLED and cuRandomForestClassifier is not None and isinstance(model, cuRandomForestClassifier):
        return cudf.DataFrame.from_pandas(X.reset_index(drop=True))
    return X

def prepare_labels(model, y):
    if GPU_ENABLED and cuRandomForestClassifier is not None and isinstance(model, cuRandomForestClassifier):
        return cudf.Series(y.reset_index(drop=True))
    return y

# Full-feature models
models_full = {
    "tree": DecisionTreeClassifier(random_state=12),
    "forest": RandomForestClassifier(n_estimators=100, random_state=12),
    "grad": HistGradientBoostingClassifier(random_state=12)
}

# PCA-reduced models
models_pca = {
    "tree": DecisionTreeClassifier(random_state=12),
    "forest": RandomForestClassifier(n_estimators=100, random_state=12),
    "grad": HistGradientBoostingClassifier(random_state=12)
}

if XGB_AVAILABLE:
    xgb_params = {"random_state": 12, "n_estimators": 100, "eval_metric": "logloss", "tree_method": "hist"}
    if GPU_ENABLED:
        xgb_params["device"] = "cuda"
    models_full["xgb"] = XGBClassifier(**xgb_params)
    models_pca["xgb"] = XGBClassifier(**xgb_params)

for name in models_full:
    models_full[name].fit(prepare_features(models_full[name], X_train), prepare_labels(models_full[name], y_train))
    models_pca[name].fit(prepare_features(models_pca[name], X_train_pca), prepare_labels(models_pca[name], y_train))

print("All available models trained.")

All models trained.


In [4]:
from sklearn.metrics import precision_score, recall_score, f1_score

def show_scores(y_true, y_pred):
    print("\tPrecision:", precision_score(y_true=y_true, y_pred=y_pred))
    print("\tRecall:", recall_score(y_true=y_true, y_pred=y_pred))
    print("\tF1:", f1_score(y_true=y_true, y_pred=y_pred))
    print()

def score_model(model, model_name, pca=False):
    print("Scores for", model_name, "(PCA reduced)" if pca else "")

    X_eval = X_val_pca if pca else X_val
    y_pred = model.predict(prepare_features(model, X_eval))
    if hasattr(y_pred, "to_pandas"):
        y_pred = y_pred.to_pandas()
    elif hasattr(y_pred, "get"):
        y_pred = y_pred.get()

    show_scores(y_val, y_pred)

## Scores for preliminary models

In [5]:
full_names = {
    "tree": "Decision Tree Classifier",
    "forest": "Random Forest Classifier",
    "grad": "Gradient Boosting Classifier",
    "xgb": "XGBoost Classifier"
}

results = []

for key, label in full_names.items():
    # Full features
    if key not in models_full:
        continue
    y_pred_full = models_full[key].predict(prepare_features(models_full[key], X_val))
    if hasattr(y_pred_full, "to_pandas"):
        y_pred_full = y_pred_full.to_pandas()
    elif hasattr(y_pred_full, "get"):
        y_pred_full = y_pred_full.get()
    print("Scores for", label)
    show_scores(y_val, y_pred_full)
    results.append([
        label,
        "Full",
        precision_score(y_val, y_pred_full),
        recall_score(y_val, y_pred_full),
        f1_score(y_val, y_pred_full)
    ])
    
    # PCA features
    y_pred_pca = models_pca[key].predict(prepare_features(models_pca[key], X_val_pca))
    if hasattr(y_pred_pca, "to_pandas"):
        y_pred_pca = y_pred_pca.to_pandas()
    elif hasattr(y_pred_pca, "get"):
        y_pred_pca = y_pred_pca.get()
    print("Scores for", label, "(PCA reduced)")
    show_scores(y_val, y_pred_pca)
    results.append([
        label,
        "PCA",
        precision_score(y_val, y_pred_pca),
        recall_score(y_val, y_pred_pca),
        f1_score(y_val, y_pred_pca)
    ])

results_df = pd.DataFrame(
    results,
    columns=["Model", "Features", "Precision", "Recall", "F1"]
)

results_df

Scores for Decision Tree Classifier
	Precision: 0.18152350081037277
	Recall: 0.4507405022537025
	F1: 0.2588159171789065

Scores for Decision Tree Classifier (PCA reduced)
	Precision: 0.09358946686812
	Recall: 0.34900193174500965
	F1: 0.14759846138135277

Scores for Random Forest Classifier
	Precision: 0.2064387179670212
	Recall: 0.46555054732775275
	F1: 0.28603926610949015

Scores for Random Forest Classifier (PCA reduced)
	Precision: 0.12114279337090564
	Recall: 0.34948486799742434
	F1: 0.17991961214933908

Scores for Gradient Boosting Classifier
	Precision: 0.09955827387020047
	Recall: 0.8018351577591758
	F1: 0.17712426435289724

Scores for Gradient Boosting Classifier (PCA reduced)
	Precision: 0.04714737103702915
	Recall: 0.8551191242755957
	F1: 0.08936742934051144



,Model,Features,Precision,Recall,F1
0,Decision Tree Classifier,Full,0.181524,0.450741,0.258816
1,Decision Tree Classifier,PCA,0.093589,0.349002,0.147598
2,Random Forest Classifier,Full,0.206439,0.465551,0.286039
3,Random Forest Classifier,PCA,0.121143,0.349485,0.179920
4,Gradient Boosting Classifier,Full,0.099558,0.801835,0.177124
5,Gradient Boosting Classifier,PCA,0.047147,0.855119,0.089367


In [6]:
results_df.sort_values(by="F1", ascending=False)

,Model,Features,Precision,Recall,F1
2,Random Forest Classifier,Full,0.206439,0.465551,0.286039
0,Decision Tree Classifier,Full,0.181524,0.450741,0.258816
3,Random Forest Classifier,PCA,0.121143,0.349485,0.179920
4,Gradient Boosting Classifier,Full,0.099558,0.801835,0.177124
1,Decision Tree Classifier,PCA,0.093589,0.349002,0.147598
5,Gradient Boosting Classifier,PCA,0.047147,0.855119,0.089367


In [7]:
best_row = results_df.sort_values(by="F1", ascending=False).iloc[0]
print("Best preliminary model:")
print(best_row)

Best preliminary model:
Model        Random Forest Classifier
Features                         Full
Precision                    0.206439
Recall                       0.465551
F1                           0.286039
Name: 2, dtype: object


In [7]:
import os
os.environ['DYLD_LIBRARY_PATH'] = '/Users/Ife/homebrew/Cellar/libomp/22.1.3/lib'

from xgboost import XGBClassifier

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/Ife/Documents/bitcoin-ransomware-classifier/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Users/Ife/Documents/bitcoin-ransomware-classifier/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot